In [7]:
import pandas as pd
import numpy as np
import os
import sklearn.utils.validation

# ==========================================
# 1. THE FIX: MANUAL PATCH FOR PYTHON 3.13
# ==========================================
if not hasattr(sklearn.utils.validation, '_is_pandas_df'):
    def _is_pandas_df(X):
        return hasattr(X, "iloc") and hasattr(X, "columns")
    sklearn.utils.validation._is_pandas_df = _is_pandas_df

# Now these imports will work
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ==========================================
# 2. LOAD PROCESSED DATA
# ==========================================
# Adjust the path to where your file is saved
file_path = os.path.join('..', 'data', 'processed', 'processed_fraud_data.csv')

if not os.path.exists(file_path):
    print(f"Error: Could not find {file_path}. Please run src/preprocessing.py first.")
else:
    df = pd.read_csv(file_path)

    # ==========================================
    # 3. PREPARE FEATURES AND TARGET
    # ==========================================
    # We remove non-numeric columns and columns that don't help prediction
    cols_to_drop = ['class', 'user_id', 'signup_time', 'purchase_time', 'device_id', 'ip_address']
    X = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
    y = df['class']

    # ==========================================
    # 4. STRATIFIED SPLIT (Requirement)
    # ==========================================
    # We split first to ensure the test set remains untouched by synthetic data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # ==========================================
    # 5. HANDLE CLASS IMBALANCE (Requirement)
    # ==========================================
    # Justification: We use SMOTE to balance the training set. 
    # This prevents the model from ignoring the fraud class due to its rarity.
    print(f"Distribution BEFORE SMOTE: {np.bincount(y_train)}")

    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    print(f"Distribution AFTER SMOTE: {np.bincount(y_train_resampled)}")

    # ==========================================
    # 6. NORMALIZATION/SCALING (Requirement)
    # ==========================================
    scaler = StandardScaler()
    X_train_resampled = scaler.fit_transform(X_train_resampled)
    X_test = scaler.transform(X_test)

    print("\nSUCCESS: Data is now split, balanced with SMOTE, and scaled.")

Distribution BEFORE SMOTE: [109568  11321]
Distribution AFTER SMOTE: [109568 109568]

SUCCESS: Data is now split, balanced with SMOTE, and scaled.
